# 🛡️ Autonomy Calibration — GRPO Training Notebook
**OpenEnv India Hackathon 2026**

This notebook fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` using **GRPO** (Group Relative Policy Optimization) via Hugging Face TRL.
Rewards come directly from the live **Autonomy Calibration Environment** API.

### ⚡ Setup
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells in order
3. After training, download the 3 plot PNGs and commit them to your repo

**Environment API:** `https://joy0021-autonomy-calibration-benchmark.hf.space/api`

## Cell 1 — Install Dependencies

In [ ]:
# Takes ~2 minutes on a fresh Colab runtime
!pip install -q trl==0.9.6 transformers==4.45.0 accelerate peft bitsandbytes requests matplotlib
print('✅ Done')

## Cell 2 — Configuration

In [ ]:
import os, json, random, requests
import torch

# ── Live Environment (your deployed HF Space) ─────────────────────────────
ENV_API = "https://joy0021-autonomy-calibration-benchmark.hf.space/api"
TASKS   = ["email_triage", "devops_incident", "financial_request"]

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-0.5B-Instruct"   # ~1 GB, fits in free T4
MAX_NEW_TOKENS = 32
NUM_STEPS      = 80    # ~15 min on T4; raise to 200 for stronger results
LEARNING_RATE  = 5e-6
BATCH_SIZE     = 2
NUM_GENERATIONS = 4   # GRPO group size

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_ID}')
print(f'Env API : {ENV_API}')

# Quick API sanity check
resp = requests.post(f'{ENV_API}/reset', json={'task': 'email_triage'}, timeout=20)
print(f'API /reset status: {resp.status_code}')
obs = resp.json()
print(f'Sample actions: {obs["available_actions"]}')

## Cell 3 — Environment Helper Functions

In [ ]:
def env_reset(task: str) -> dict:
    r = requests.post(f'{ENV_API}/reset', json={'task': task}, timeout=20)
    r.raise_for_status()
    return r.json()

def env_step(action_type: str) -> dict:
    r = requests.post(f'{ENV_API}/step', json={'type': action_type}, timeout=20)
    r.raise_for_status()
    return r.json()

def build_prompt(obs: dict) -> str:
    """Convert an observation dict into a clean LLM prompt."""
    actions = obs.get('available_actions', [])
    situation = obs.get('prompt', '')
    return (
        f"You are an AI agent making real-world decisions.\n"
        f"Situation: {situation}\n"
        f"Available actions: {actions}\n"
        f"Reply with ONLY one action name from the list above:"
    )

def extract_action(text: str, available: list) -> str:
    """Parse the model output to find the first valid action name."""
    for a in available:
        if a in text:
            return a
    return random.choice(available)  # fallback

print('✅ Environment helpers ready')

## Cell 4 — Load Model + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('✅ Model ready')

## Cell 5 — Build Training Dataset

We collect prompt snapshots from the live environment across all 3 tasks.

In [ ]:
from datasets import Dataset

records = []
for task in TASKS:
    for _ in range(15):   # 15 episodes per task = 45 total prompts
        try:
            obs = env_reset(task)
            records.append({
                'prompt': build_prompt(obs),
                'task_id': task,
                'available_actions': json.dumps(obs.get('available_actions', []))
            })
        except Exception as e:
            print(f'Warning: {e}')

train_dataset = Dataset.from_list(records)
print(f'✅ Dataset: {len(train_dataset)} prompts across {len(TASKS)} tasks')
print(train_dataset)

## Cell 6 — GRPO Reward Function

This is the core of training. For each model completion, we call the live environment and get a real reward signal.

In [ ]:
def autonomy_reward(completions, prompts=None, task_id=None, available_actions=None, **kwargs):
    """
    GRPO reward function.
    Parses the model's generated text, sends the action to the
    live environment, and returns the environment reward.
    """
    rewards = []
    n = len(completions)
    task_ids   = task_id if task_id else ['email_triage'] * n
    avail_list = available_actions if available_actions else ['[]'] * n

    for i, completion in enumerate(completions):
        # Completions from TRL can be list of dicts or plain strings
        if isinstance(completion, list):
            text = completion[0].get('content', '') if completion else ''
        else:
            text = str(completion)

        avail = json.loads(avail_list[i]) if isinstance(avail_list[i], str) else avail_list[i]
        action = extract_action(text, avail)

        try:
            env_reset(task_ids[i])           # fresh episode for each evaluation
            result = env_step(action)
            reward = result.get('reward', {}).get('value', 0.01)
            rewards.append(float(reward))
        except Exception:
            rewards.append(0.01)             # API failure → minimum reward

    return rewards

print('✅ Reward function defined')
print('   Each completion → live env call → reward in [0.01, 0.99]')

## Cell 7 — Configure & Run GRPO Training

⏱️ Expected time on T4: ~15 minutes for 80 steps.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir          = './grpo_checkpoints',
    max_steps           = NUM_STEPS,
    per_device_train_batch_size = BATCH_SIZE,
    learning_rate       = LEARNING_RATE,
    max_new_tokens      = MAX_NEW_TOKENS,
    max_prompt_length   = 400,
    num_generations     = NUM_GENERATIONS,
    temperature         = 0.7,
    logging_steps       = 5,
    save_steps          = 40,
    gradient_accumulation_steps = 2,
    report_to           = 'none',
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = autonomy_reward,
    args             = config,
    train_dataset    = train_dataset,
)

print(f'🚀 Starting GRPO training ({NUM_STEPS} steps)...')
train_result = trainer.train()
print('\n✅ Training complete!')
print(f'   Total steps: {train_result.global_step}')
print(f'   Train loss:  {train_result.training_loss:.4f}')

## Cell 8 — Plot Training Curves and Save as PNG

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

log = trainer.state.log_history

steps   = [e['step']   for e in log if 'loss' in e]
losses  = [e['loss']   for e in log if 'loss' in e]
r_steps = [e['step']   for e in log if 'reward' in e]
rewards = [e['reward'] for e in log if 'reward' in e]

# ─── Plot 1: Loss curve ────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, color='#E74C3C', linewidth=2)
plt.title('GRPO Policy Loss During Training')
plt.xlabel('Training Step')
plt.ylabel('Policy Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.close()
print('✅ Saved loss_curve.png')

# ─── Plot 2: Reward curve ─────────────────────────────────────────────────
BASELINE = 0.611   # rule-based baseline (email 0.46, devops 0.68, fin 0.69)

plt.figure(figsize=(10, 5))
if rewards:
    plt.plot(r_steps, rewards, color='#27AE60', linewidth=2, label='GRPO Reward')
plt.axhline(y=BASELINE, color='#3498DB', linestyle='--',
            linewidth=2, label=f'Baseline ({BASELINE})')
plt.title('Episode Reward During GRPO Training')
plt.xlabel('Training Step')
plt.ylabel('Episode Reward (0.01–0.99)')
plt.ylim(0.0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.close()
print('✅ Saved reward_curve.png')

## Cell 9 — Evaluate Trained Agent vs Baseline

In [ ]:
def eval_model(task, n_episodes=5):
    scores = []
    for ep in range(n_episodes):
        obs  = env_reset(task)
        done = False
        ep_score = 0.0
        step = 0
        while not done and step < 6:
            prompt = build_prompt(obs)
            inputs = tokenizer(prompt, return_tensors='pt',
                               truncation=True, max_length=400).to(DEVICE)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                     temperature=0.1, do_sample=True,
                                     pad_token_id=tokenizer.eos_token_id)
            text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True)
            action  = extract_action(text, obs.get('available_actions', []))
            result  = env_step(action)
            ep_score = result.get('info', {}).get('episode_score', ep_score)
            obs     = result.get('observation', {})
            done    = result.get('done', False)
            step   += 1
        scores.append(ep_score)
    return scores

BASELINE_SCORES = {'email_triage': 0.46, 'devops_incident': 0.68, 'financial_request': 0.69}
trained_scores  = {}

print('Evaluating trained agent...')
for task in TASKS:
    sc = eval_model(task, n_episodes=5)
    trained_scores[task] = sc
    print(f'  {task}: avg={sum(sc)/len(sc):.3f}  baseline={BASELINE_SCORES[task]}')

# ─── Plot 3: Baseline vs Trained bar chart ────────────────────────────────
labels   = [t.replace('_', '\n') for t in TASKS]
baseline = [BASELINE_SCORES[t] for t in TASKS]
trained  = [sum(trained_scores[t])/len(trained_scores[t]) for t in TASKS]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, baseline, width, label='Rule-Based Baseline', color='#95A5A6')
b2 = ax.bar(x + width/2, trained,  width, label='GRPO Trained Agent',  color='#27AE60')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.2f}',
            ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Final Score Comparison: Baseline vs GRPO Trained Agent')
ax.set_ylabel('Average Episode Reward (0.01–0.99)')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('baseline_vs_trained.png', dpi=150)
plt.close()
print('✅ Saved baseline_vs_trained.png')

## Cell 10 — Download Your Plots

Run this cell to download the 3 plots to your computer.
Then commit them to your repo: `git add *.png && git commit -m 'Real training plots from Colab GRPO run'`

In [ ]:
from google.colab import files
for f in ['loss_curve.png', 'reward_curve.png', 'baseline_vs_trained.png']:
    files.download(f)
    print(f'⬇️  Downloaded {f}')

## Cell 11 — (Optional) Push Trained Model to HF Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN = 'your_new_hf_token_here'   # paste a NEW write token here
HF_REPO  = 'JOY0021/autonomy-grpo-agent'

if HF_TOKEN != 'your_new_hf_token_here':
    login(token=HF_TOKEN)
    trainer.model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f'✅ Model pushed → https://huggingface.co/{HF_REPO}')
else:
    print('⚠️  Add your HF token to push the model. Skipping.')